In [1]:
import requests
import torch

from PIL import Image
from transformers import AutoProcessor, AutoModelForCausalLM
from pathlib import Path

import pandas as pd

In [2]:
FILE_NAME = Path("furniture_amazon_dataset_sample.csv")

if not FILE_NAME.exists():
    raise FileNotFoundError(f"Download csv from discord bozo and put it at: spaiced/miguellib/datasets/{FILE_NAME}")

amazon_df = pd.read_csv(FILE_NAME)
display(amazon_df)

,asin,url,title,brand,price,availability,categories,primary_image,images,upc,...,color,material,style,important_information,product_overview,about_item,description,specifications,uniq_id,scraped_at
0,B0CJHKVG6P,https://www.amazon.com/dp/B0CJHKVG6P,"GOYMFK 1pc Free Standing Shoe Rack, Multi-laye...",GOYMFK,$24.99,Only 13 left in stock - order soon.,"['Home & Kitchen', 'Storage & Organization', '...",https://m.media-amazon.com/images/I/416WaLx10j...,['https://m.media-amazon.com/images/I/416WaLx1...,NaN,...,White,Metal,Modern,[],"[{'Brand': ' GOYMFK '}, {'Color': ' White '}, ...",['Multiple layers: Provides ample storage spac...,"multiple shoes, coats, hats, and other items E...","['Brand: GOYMFK', 'Color: White', 'Material: M...",02593e81-5c09-5069-8516-b0b29f439ded,2024-02-02 15:15:08
1,B0B66QHB23,https://www.amazon.com/dp/B0B66QHB23,"subrtex Leather ding Room, Dining Chairs Set o...",subrtex,NaN,NaN,"['Home & Kitchen', 'Furniture', 'Dining Room F...",https://m.media-amazon.com/images/I/31SejUEWY7...,['https://m.media-amazon.com/images/I/31SejUEW...,NaN,...,Black,Sponge,Black Rubber Wood,[],NaN,['【Easy Assembly】: Set of 2 dining room chairs...,subrtex Dining chairs Set of 2,"['Brand: subrtex', 'Color: Black', 'Product Di...",5938d217-b8c5-5d3e-b1cf-e28e340f292e,2024-02-02 15:15:09
2,B0BXRTWLYK,https://www.amazon.com/dp/B0BXRTWLYK,Plant Repotting Mat MUYETOL Waterproof Transpl...,MUYETOL,$5.98,In Stock,"['Patio, Lawn & Garden', 'Outdoor Décor', 'Doo...",https://m.media-amazon.com/images/I/41RgefVq70...,['https://m.media-amazon.com/images/I/41RgefVq...,NaN,...,Green,Polyethylene,Modern,[],"[{'Brand': ' MUYETOL '}, {'Size': ' 26.8*26.8 ...","['PLANT REPOTTING MAT SIZE: 26.8"" x 26.8"", squ...",NaN,"['Brand: MUYETOL', 'Size: 26.8*26.8', 'Item We...",b2ede786-3f51-5a45-9a5b-bcf856958cd8,2024-02-02 15:15:09
3,B0C1MRB2M8,https://www.amazon.com/dp/B0C1MRB2M8,"Pickleball Doormat, Welcome Doormat Absorbent ...",VEWETOL,$13.99,Only 10 left in stock - order soon.,"['Patio, Lawn & Garden', 'Outdoor Décor', 'Doo...",https://m.media-amazon.com/images/I/61vz1Igler...,['https://m.media-amazon.com/images/I/61vz1Igl...,NaN,...,A5589,Rubber,Modern,[],"[{'Brand': ' VEWETOL '}, {'Size': ' 16*24INCH ...","['Specifications: 16x24 Inch ', "" High-Quality...",The decorative doormat features a subtle textu...,"['Brand: VEWETOL', 'Size: 16*24INCH', 'Materia...",8fd9377b-cfa6-5f10-835c-6b8eca2816b5,2024-02-02 15:15:10
4,B0CG1N9QRC,https://www.amazon.com/dp/B0CG1N9QRC,JOIN IRON Foldable TV Trays for Eating Set of ...,JOIN IRON Store,$89.99,Usually ships within 5 to 6 weeks,"['Home & Kitchen', 'Furniture', 'Game & Recrea...",https://m.media-amazon.com/images/I/41p4d4VJnN...,['https://m.media-amazon.com/images/I/41p4d4VJ...,NaN,...,Grey Set of 4,Iron,X Classic Style,[],NaN,['Includes 4 Folding Tv Tray Tables And one Co...,Set of Four Folding Trays With Matching Storag...,"['Brand: JOIN IRON', 'Shape: Rectangular', 'In...",bdc9aa30-9439-50dc-8e89-213ea211d66a,2024-02-02 15:15:11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307,B08SLPBC36,https://www.amazon.com/dp/B08SLPBC36,Lexicon Victoria Saddle Wood Bar Stools (Set o...,Lexicon,$58.99,Only 7 left in stock (more on the way).,"['Home & Kitchen', 'Furniture', 'Game & Recrea...",https://m.media-amazon.com/images/I/41CPL03Y-W...,['https://m.media-amazon.com/images/I/41CPL03Y...,NaN,...,Black Sand,Wood,Contemporary,[],NaN,"['Frame Material: Wood ', ' Set includes two (...",With a country flair and a deep black sand fin...,"['Product Dimensions: 18""D x 15.5""W x 29""H', '...",d3b681ac-6195-5c9b-8125-98b6425829f4,2024-02-02 18:56:50
308,B09KN5ZTXC,https://www.amazon.com/dp/B09KN5ZTXC,ANZORG Behind Door Hanging Kids Shoes Organize...,ANZORG Store,$9.99,Only 14 left in stock - order soon.,"['Home & Kitchen', 'Storage & Organization', '...",https://m.media-amazon.com/images/I/31qQ2tZPv-...,['https://m.media-amazon.com/images/I/31qQ2tZP...,NaN,...,12 Pockets,Non Woven Fabric,

In [7]:
# Identify image column to enhance (let's stick with primary image for now)
device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model = AutoModelForCausalLM.from_pretrained("microsoft/Florence-2-large", torch_dtype=torch_dtype, trust_remote_code=True).to(device)
processor = AutoProcessor.from_pretrained("microsoft/Florence-2-large", trust_remote_code=True)

# prompt = (
#     "<DETAILED_CAPTION>"
# )

prompt = "<DETAILED_CAPTION>"

url = amazon_df["primary_image"].iloc[0]
# url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/car.jpg?download=true"
image = Image.open(requests.get(url, stream=True).raw)

inputs = processor(text=prompt, images=image, return_tensors="pt").to(device, torch_dtype)

generated_ids = model.generate(
    input_ids=inputs["input_ids"],
    pixel_values=inputs["pixel_values"],
    max_new_tokens=256,
    num_beams=3,
    do_sample=False
)
generated_text = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]

parsed_answer = processor.post_process_generation(generated_text, task="<DETAILED_CAPTION>", image_size=(image.width, image.height))

print(parsed_answer)

{'<DETAILED_CAPTION>': 'The image shows a white shoe rack with several pairs of shoes neatly arranged on it. On the left side of the rack is a wooden table with a flower vase on top. In the background, there is a wall and a door.'}


## run on gpu

In [ ]:
if not torch.cuda.is_available:
    raise ValueError("get torch")

device = "cuda:0"
dtype = torch.float16
model_id = "microsoft/Florence-2-large"

model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=dtype, trust_remote_code=True).to(device)
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)

TASK = "<DETAILED_CAPTION>"
INSTR = ("Describe this furniture item for an e-commerce listing. "
         "Include furniture type, style, materials, and dominant color(s). "
         "Do not guess brand/model/dimensions.")

prompt = f"{TASK}"

def fetch_image(url: str) -> Image.Image:
    r = requests.get(url, timeout=20)
    r.raise_for_status()
    return Image.open(BytesIO(r.content)).convert("RGB")

def run_batch(urls):
    # 1) Download in parallel
    images = [None] * len(urls)
    with ThreadPoolExecutor(max_workers=16) as ex:
        futs = {ex.submit(fetch_image, u): i for i, u in enumerate(urls)}
        for fut in as_completed(futs):
            i = futs[fut]
            images[i] = fut.result()

    # 2) Batched processor call (same prompt repeated)
    texts = [prompt] * len(images)
    inputs = processor(text=texts, images=images, return_tensors="pt", padding=True).to(device, dtype)

    # 3) Generate (batched)
    with torch.inference_mode():
        gen_ids = model.generate(
            input_ids=inputs["input_ids"],
            pixel_values=inputs["pixel_values"],
            max_new_tokens=256,
            num_beams=3,
            do_sample=False,
        )

    # 4) Decode each
    out = processor.batch_decode(gen_ids, skip_special_tokens=False)

    # 5) Postprocess each with its own image size
    results = []
    for txt, im in zip(out, images):
        parsed = processor.post_process_generation(
            txt, task=TASK, image_size=(im.width, im.height)
        )
        results.append(parsed)
    return results

# Example driver
def chunked(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

urls = amazon_df["primary_image"].tolist()

BATCH_SIZE = 4 if torch.cuda.is_available() else 1

all_results = []
for batch_urls in chunked(urls, BATCH_SIZE):
    all_results.extend(run_batch(batch_urls))
